# Download Street View Images for Building Footprints

For each building footprint, this notebook finds nearby Google Street View panoramas, keeps only the ones with a clear view of the building, and downloads an image from each one, pointed at the building.

**Before you start**
1. Set your Google Maps API key (see the README): `export GOOGLE_MAPS_API_KEY="your-key"` before launching Jupyter.
2. Edit the **INPUTS** cell below.

**Inputs**
- **All footprints** (`ALL_FOOTPRINTS_PATH`, GeoJSON): every building footprint in and around the study area. These are the obstacles for the line-of-sight checks, so this layer should extend ~100 m+ beyond the buildings you want images of.
- **Needed footprints** (`NEEDED_FOOTPRINTS_PATH`, GeoJSON): the buildings you want images of. Can be the same file as the all-footprints layer.
- Both files must share the same unique ID column (`ID_COL`).

**Download method** (`DOWNLOAD_METHOD`)
- `'api'` (default, recommended): official Google Static Street View API. Billed per image.
- `'brails'`: multi-image modification of the BRAILS++ image download. See the README.

**How it works**
1. For each needed footprint, sample `N_SEARCH_POINTS` points on a circle of `SEARCH_RADIUS_M` around its centroid and look up the nearest panorama to each.
2. Skip panoramas that are duplicates, too far from the building, or whose line of sight (from the panorama to the nearest edge of the footprint) crosses or passes too close to another footprint.
3. Download a view from each remaining panorama, with the camera pointed at the footprint's centroid.
4. Results are saved to a checkpoint after every footprint, so the run can be stopped and **re-run to resume**.

**Coordinates:** distance calculations use a UTM zone in meters, chosen automatically from the footprint locations, so this works anywhere. Lat/lon (EPSG:4326) is used for the Street View requests.

## INPUTS: edit this cell

In [ ]:
# --- Footprint files (GeoJSON) ----------------------------------------------
# All footprints: used as line-of-sight obstacles. Should cover the needed
# footprints plus at least ~100 m around them.
ALL_FOOTPRINTS_PATH = 'example_footprints.geojson'

# Needed footprints: the buildings you want images of (can be the same file).
NEEDED_FOOTPRINTS_PATH = 'example_footprints.geojson'

# Unique building ID column, present in BOTH files with matching values.
ID_COL = 'FootprintID'

# Only process the first N needed footprints (useful for a quick test). None = all images.
MAX_FOOTPRINTS = 3



### Acceptable to leave all flags below here unchanged 

# --- Download method ---------------------------------------------------------
# 'api'    : Google Static Street View API (default, recommended; billed per image)
# 'brails' : multi-image modification of the BRAILS++ image download (see README)
DOWNLOAD_METHOD = 'api'

# --- Output ------------------------------------------------------------------
# Images go to OUTPUT_DIR/images/; checkpoint.csv and image_info.csv go to OUTPUT_DIR/.
OUTPUT_DIR = 'output'

# Save a diagram for each building to OUTPUT_DIR/diagrams/ showing the requested
# points, panoramas, and lines of sight, colored by download_tag. Slower; needs matplotlib.
SAVE_DIAGRAMS = True
# Diagram lines go from each panorama to: 'centroid' (where the camera points) or
# 'nearest' (closest point on the footprint edge, used by the line-of-sight filters).
DIAGRAM_LINES_TO = 'centroid'

### The following settings (search and filter, field of view)were established in Lochhead et al. (2026) and are generally appropriate for most single-family or small multi-family buildings. 
### These settings may need to be adjusted for larger buildings, townhouses, etc. 

# --- Search and filter settings ------------------------------------------------
SEARCH_RADIUS_M = 15        # radius of the sample points around each centroid
N_SEARCH_POINTS = 8         # number of sample points around each centroid
NEIGHBOR_BUFFER_M = 100     # only footprints within this distance are checked as obstacles
MAX_DIST_TO_FTPT_M = 33     # skip panoramas farther than this from the footprint
MIN_LOS_CLEARANCE_M = 0.66  # skip if the line of sight passes closer than this to another footprint
MAX_WORKERS = 10            # footprints processed in parallel

# Field of view (degrees) chosen by distance from panorama to footprint:
# use the first entry whose distance limit is greater than the actual distance.
FOV_BY_DISTANCE_M = [(7, 80), (14, 60), (16, 50)]   # (max distance in m, fov)
FOV_DEFAULT = 40                                     # farther than all limits above

## Setup

In [3]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString

import streetview_images as svi

# API key is read from an environment variable so it never gets saved in the notebook.
API_KEY = os.environ.get('GOOGLE_MAPS_API_KEY', '')
if not API_KEY:
    raise RuntimeError("Set the GOOGLE_MAPS_API_KEY environment variable first (see README).")

assert DOWNLOAD_METHOD in ('api', 'brails'), "DOWNLOAD_METHOD must be 'api' or 'brails'"

CRS_GEO = 'EPSG:4326'  # WGS84 lat/lon, used for Street View requests

image_dir = os.path.join(OUTPUT_DIR, 'images')
os.makedirs(image_dir, exist_ok=True)
diagram_dir = os.path.join(OUTPUT_DIR, 'diagrams')
if SAVE_DIAGRAMS:
    os.makedirs(diagram_dir, exist_ok=True)
checkpoint_file = os.path.join(OUTPUT_DIR, 'checkpoint.csv')
final_file = os.path.join(OUTPUT_DIR, 'image_info.csv')


def read_footprints(path):
    # Read a GeoJSON footprint file and drop empty geometries.
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        raise ValueError(f"{path} has no CRS defined. Set it with gdf.set_crs(...) before use.")
    if ID_COL not in gdf.columns:
        raise KeyError(f"{path} is missing the ID column '{ID_COL}'.")
    return gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].reset_index(drop=True)


def choose_fov(distance_m):
    for max_dist, fov in FOV_BY_DISTANCE_M:
        if distance_m < max_dist:
            return fov
    return FOV_DEFAULT

## Load footprints

In [4]:
ftpts_all = read_footprints(ALL_FOOTPRINTS_PATH)
needed_rows = read_footprints(NEEDED_FOOTPRINTS_PATH)

# Optional: filter needed_rows here (e.g. by city or building type).
# needed_rows = needed_rows[needed_rows['some_column'] == 'some_value']

if MAX_FOOTPRINTS is not None:
    needed_rows = needed_rows.head(MAX_FOOTPRINTS)

# Project both layers to a UTM zone (meters) chosen from the data location.
CRS_METRIC = ftpts_all.estimate_utm_crs()
ftpts_all = ftpts_all.to_crs(CRS_METRIC)
needed_rows = needed_rows.to_crs(CRS_METRIC)
_ = ftpts_all.sindex  # build the spatial index once, before threads use it

print(f"Metric CRS: {CRS_METRIC.name}")
print(f"All footprints: {len(ftpts_all)}")
print(f"Needed footprints: {len(needed_rows)}")

# Each building's own footprint is excluded from the obstacles by ID. If IDs don't
# match between the two files, a building's own footprint would block its views.
missing = ~needed_rows[ID_COL].isin(ftpts_all[ID_COL])
if missing.any():
    print(f"WARNING: {missing.sum()} needed footprints have no matching '{ID_COL}' in the all-footprints file.")

Metric CRS: WGS 84 / UTM zone 10N
All footprints: 1579
Needed footprints: 3


## Per-footprint processing

Each sample point gets one row in the output with a `download_tag` (meanings are listed in the README).

In [5]:
def process_row(row):
    records = []
    footprint = row.geometry      # in CRS_METRIC
    boundary = footprint.boundary
    centroid_geo = gpd.GeoSeries([footprint.centroid], crs=CRS_METRIC).to_crs(CRS_GEO).iloc[0]
    sample_points = svi.generate_surrounding_points(footprint.centroid, CRS_METRIC,
                                                    radius_m=SEARCH_RADIUS_M, n_points=N_SEARCH_POINTS)

    # Other footprints within NEIGHBOR_BUFFER_M (the possible obstacles)
    nearby = ftpts_all.iloc[ftpts_all.sindex.query(footprint.buffer(NEIGHBOR_BUFFER_M))]
    other_footprints = nearby[nearby[ID_COL] != row[ID_COL]]

    seen_pano_ids = set()
    diagram_panos = []  # for the optional diagram: pano location, line-of-sight target, tag

    for i, pt in enumerate(sample_points):
        rec = {ID_COL: row[ID_COL], 'view': i, 'pano_id': 'NA', 'distance_to_ftpt': 'NA',
               'min_line_of_sight_clearance': 'NA', 'image_year': 'NA', 'download_tag': None}
        records.append(rec)

        try:
            pano = svi.get_pano((pt['lat'], pt['lon']), API_KEY)
        except ValueError:
            rec['download_tag'] = 'no_pano_found'
            continue

        pano_id = pano['pano_id']
        rec['pano_id'] = pano_id
        rec['image_year'] = pano['date']

        pano_lat, pano_lon = pano['location']
        pano_pt = gpd.GeoSeries([Point(pano_lon, pano_lat)], crs=CRS_GEO).to_crs(CRS_METRIC).iloc[0]

        if pano_id in seen_pano_ids:
            rec['download_tag'] = 'duplicate_pano'
            continue
        seen_pano_ids.add(pano_id)

        # Distance from the panorama to the nearest point on the footprint edge
        distance = pano_pt.distance(boundary)
        rec['distance_to_ftpt'] = distance

        # Line of sight: panorama -> nearest point on the footprint edge
        nearest = boundary.interpolate(boundary.project(pano_pt))
        line = LineString([pano_pt, nearest])
        clearance = other_footprints.distance(line).min() if len(other_footprints) else float('inf')
        rec['min_line_of_sight_clearance'] = clearance
        diagram_panos.append({'point': pano_pt, 'target': nearest, 'rec': rec})

        if other_footprints.intersects(line).any():
            rec['download_tag'] = 'intersects'
            continue
        if distance > MAX_DIST_TO_FTPT_M:
            rec['download_tag'] = 'too_far_away'
            continue
        if clearance < MIN_LOS_CLEARANCE_M:
            rec['download_tag'] = 'line_of_sight_too_close'
            continue

        # Passed all filters: download, with the camera pointed at the centroid
        heading = svi.compute_heading(pano_lat, pano_lon, centroid_geo.y, centroid_geo.x)
        fov = choose_fov(distance)
        filename = os.path.join(image_dir, f"{row[ID_COL]}_view{i}.jpg")

        try:
            if DOWNLOAD_METHOD == 'api':
                svi.download_street_view_api(pano_id, heading=heading, api_key=API_KEY,
                                             filename=filename, fov=fov)
            else:
                svi.download_street_view_brails(pano_id, heading=heading, filename=filename,
                                                fov=fov, pitch=0, zoom=4, output_size=(640, 640))
            rec['download_tag'] = 'downloaded'
        except Exception as e:
            print(f"[DOWNLOAD FAIL] {ID_COL}={row[ID_COL]}, pano={pano_id}: {e}")
            rec['download_tag'] = 'download_failed'

        time.sleep(0.1)

    if SAVE_DIAGRAMS:
        panos = [{'point': p['point'], 'target': p['target'], 'tag': p['rec']['download_tag']}
                 for p in diagram_panos]
        svi.plot_download_diagram(footprint, other_footprints, sample_points, panos, CRS_METRIC,
                                  title=f"{ID_COL} = {row[ID_COL]}",
                                  filename=os.path.join(diagram_dir, f"{row[ID_COL]}.png"),
                                  lines_to=DIAGRAM_LINES_TO)

    return pd.DataFrame(records)

## Checkpoint / resume

Footprints already in `checkpoint.csv` are skipped. Delete `checkpoint.csv` to start over (for example, after changing the inputs or filter settings).

In [6]:
if os.path.exists(checkpoint_file):
    results = [pd.read_csv(checkpoint_file)]
    done = set(results[0][ID_COL])
else:
    results = []
    done = set()

todo = needed_rows[~needed_rows[ID_COL].isin(done)]
print(f"Already done: {len(done)}, remaining: {len(todo)}")

Already done: 0, remaining: 3


## Run downloads

Safe to interrupt: re-run the checkpoint cell and this cell to resume. Footprints that raise an error are not checkpointed, so they are retried on the next run.

In [7]:
start = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_row, row): row[ID_COL] for _, row in todo.iterrows()}

    for i, future in enumerate(as_completed(futures), start=1):
        try:
            results.append(future.result())
        except Exception as e:  # one bad footprint shouldn't stop the run; it is retried next time
            print(f"[ROW FAIL] {ID_COL}={futures[future]}: {e}")
        if results:
            pd.concat(results).to_csv(checkpoint_file, index=False)
        elapsed = time.time() - start
        print(f"{i} / {len(todo)} completed | {elapsed/60:.1f} min elapsed | ~{elapsed/i*(len(todo)-i)/60:.1f} min left")

final_df = pd.concat(results) if results else pd.DataFrame()
final_df.to_csv(final_file, index=False)
print(f"Saved {len(final_df)} rows to {final_file}")

1 / 3 completed | 0.0 min elapsed | ~0.1 min left
2 / 3 completed | 0.0 min elapsed | ~0.0 min left
3 / 3 completed | 0.0 min elapsed | ~0.0 min left
Saved 24 rows to output_goog/image_info.csv


## Summary

In [8]:
final_df['download_tag'].value_counts()

download_tag
downloaded        12
duplicate_pano     8
intersects         3
too_far_away       1
Name: count, dtype: int64